### Imports

In [2]:
from datetime import datetime, timezone
from pathlib import Path
import gzip
import json

import pandas as pd
import requests


BASE_URL = "https://fantasy.premierleague.com/api/"

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

### Data


In [3]:
session = requests.Session()

session.headers.update({
    "User-Agent": "FPL-Predictor/1.0"
})


def get_fpl_data(endpoint: str) -> dict | list:
    """
    Download JSON data from an FPL API endpoint.

    Parameters
    ----------
    endpoint:
        Endpoint relative to the FPL API base URL,
        for example 'bootstrap-static/'.

    Returns
    -------
    dict or list
        Parsed JSON response.
    """
    url = f"{BASE_URL}{endpoint}"

    try:
        response = session.get(url, timeout=30)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            f"Unable to download FPL data from {url}"
        ) from exc

    try:
        return response.json()
    except requests.JSONDecodeError as exc:
        raise RuntimeError(
            f"FPL returned an invalid JSON response from {url}"
        ) from exc

In [4]:
bootstrap = get_fpl_data("bootstrap-static/")
fixtures_raw = get_fpl_data("fixtures/")

print("Bootstrap sections:")
print(list(bootstrap.keys()))

print(f"\nNumber of fixtures: {len(fixtures_raw)}")

Bootstrap sections:
['chips', 'events', 'game_settings', 'game_config', 'phases', 'teams', 'total_players', 'element_stats', 'element_types', 'elements']

Number of fixtures: 380


In [5]:
teams = (
    pd.DataFrame(bootstrap["teams"])
    .rename(columns={
        "id": "team_id",
        "name": "team_name",
        "short_name": "team_short_name",
    })
)

positions = (
    pd.DataFrame(bootstrap["element_types"])
    .rename(columns={
        "id": "element_type",
        "singular_name": "position_name",
        "singular_name_short": "position",
    })
)

display(
    teams[
        [
            "team_id",
            "team_name",
            "team_short_name",
            "strength",
            "strength_attack_home",
            "strength_attack_away",
            "strength_defence_home",
            "strength_defence_away",
        ]
    ]
)

display(
    positions[
        [
            "element_type",
            "position_name",
            "position",
            "squad_select",
            "squad_min_play",
            "squad_max_play",
        ]
    ]
)

,team_id,team_name,team_short_name,strength,strength_attack_home,strength_attack_away,strength_defence_home,strength_defence_away
0,1,Arsenal,ARS,None,0,0,0,0
1,2,Aston Villa,AVL,None,0,0,0,0
2,3,Bournemouth,BOU,None,0,0,0,0
3,4,Brentford,BRE,None,0,0,0,0
4,5,Brighton,BHA,None,0,0,0,0
5,6,Chelsea,CHE,None,0,0,0,0
6,7,Coventry City,COV,None,0,0,0,0
7,8,Crystal Palace,CRY,None,0,0,0,0
8,9,Everton,EVE,None,0,0,0,0
9,10,Fulham,FUL,None,0,0,0,0


,element_type,position_name,position,squad_select,squad_min_play,squad_max_play
0,1,Goalkeeper,GKP,2,1,1
1,2,Defender,DEF,5,3,5
2,3,Midfielder,MID,5,2,5
3,4,Forward,FWD,3,1,3


In [6]:
players = pd.DataFrame(bootstrap["elements"])

players = (
    players
    .merge(
        teams[
            [
                "team_id",
                "team_name",
                "team_short_name",
            ]
        ],
        left_on="team",
        right_on="team_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        positions[
            [
                "element_type",
                "position_name",
                "position",
            ]
        ],
        on="element_type",
        how="left",
        validate="many_to_one",
    )
)

# FPL stores prices as integer tenths of a million:
# 45 means £4.5m, 100 means £10.0m, etc.
players["price"] = players["now_cost"] / 10

players["full_name"] = (
    players["first_name"].str.strip()
    + " "
    + players["second_name"].str.strip()
)

In [7]:
numeric_columns = [
    "selected_by_percent",
    "form",
    "points_per_game",
    "value_form",
    "value_season",
    "influence",
    "creativity",
    "threat",
    "ict_index",
    "expected_goals",
    "expected_assists",
    "expected_goal_involvements",
    "expected_goals_conceded",
]

for column in numeric_columns:
    if column in players.columns:
        players[column] = pd.to_numeric(
            players[column],
            errors="coerce",
        )

In [8]:
preferred_player_columns = [
    "id",
    "full_name",
    "web_name",
    "team_name",
    "team_short_name",
    "position",
    "price",
    "status",
    "chance_of_playing_next_round",
    "news",
    "selected_by_percent",
    "total_points",
    "points_per_game",
    "form",
    "minutes",
    "starts",
    "goals_scored",
    "assists",
    "clean_sheets",
    "goals_conceded",
    "saves",
    "bonus",
    "bps",
    "expected_goals",
    "expected_assists",
    "expected_goal_involvements",
    "expected_goals_conceded",
]

available_player_columns = [
    column
    for column in preferred_player_columns
    if column in players.columns
]

players = players[available_player_columns].copy()

display(players.head(10))

,id,full_name,web_name,team_name,team_short_name,position,price,status,chance_of_playing_next_round,news,selected_by_percent,total_points,points_per_game,form,minutes,starts,goals_scored,assists,clean_sheets,goals_conceded,saves,bonus,bps,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded
0,1,David Raya Martín,Raya,Arsenal,ARS,GKP,6.0,a,NaN,,29.0,162,4.4,0.0,3330,37,0,0,19,26,60,11,633,0.00,0.07,0.07,27.56
1,2,Kepa Arrizabalaga Revuelta,Arrizabalaga,Arsenal,ARS,GKP,5.0,a,NaN,,0.3,2,2.0,0.0,90,1,0,0,0,1,2,0,11,0.00,0.00,0.00,0.98
2,3,Illan Meslier,Meslier,Arsenal,ARS,GKP,5.0,a,NaN,,0.1,0,0.0,0.0,0,0,11,0,0,0,0,0,11,0.00,0.00,0.00,0.00
3,4,Gabriel dos Santos Magalhães,Gabriel,Arsenal,ARS,DEF,8.0,a,NaN,,23.5,209,6.5,0.0,2750,30,3,5,18,20,0,30,724,2.94,1.75,4.69,22.01
4,5,Jurriën Timber,J.Timber,Arsenal,ARS,DEF,6.5,i,0.0,Groin injury - Expected back 21 Aug,1.0,149,5.0,0.0,2452,28,3,6,13,20,0,9,532,4.71,1.53,6.24,17.45
5,6,William Saliba,Saliba,Arsenal,ARS,DEF,6.0,i,0.0,Back injury - Unknown return date,0.5,137,4.4,0.0,2614,30,1,0,15,20,0,12,581,0.88,1.21,2.09,20.41
6,7,Myles Lewis-Skelly,Lewis-Skelly,Arsenal,ARS,MID,5.5,a,NaN,,0.3,29,1.4,0.0,697,5,0,0,2,8,0,0,110,0.10,0.20,0.30,9.01
7,8,Riccardo Calafiori,Calafiori,Arsenal,ARS,DEF,5.5,a,NaN,,11.7,109,4.2,0.0,1697,22,1,2,13,8,0,6,389,3.36,0.76,4.12,9.72
8,9,Piero Hincapié,Hincapie,Arsenal,ARS,DEF,5.5,a,NaN,,6.2,87,3.5,0.0,1787,20,1,2,5,19,0,7,335,0.36,1.67,2.03,17.53
9,10,Benjamin White,White,Arsenal,ARS,DEF,5.5,i,0.0,Knee injury - Expected back 21 Aug,0.0,45,3.8,0.0,699,9,0,1,5,6,0,3,181,0.45,0.69,1.14,8.68


### Fixtures

In [9]:
fixtures = pd.DataFrame(fixtures_raw)

home_teams = teams[
    ["team_id", "team_name", "team_short_name"]
].rename(columns={
    "team_id": "team_h",
    "team_name": "home_team",
    "team_short_name": "home_team_short",
})

away_teams = teams[
    ["team_id", "team_name", "team_short_name"]
].rename(columns={
    "team_id": "team_a",
    "team_name": "away_team",
    "team_short_name": "away_team_short",
})

fixtures = (
    fixtures
    .merge(
        home_teams,
        on="team_h",
        how="left",
        validate="many_to_one",
    )
    .merge(
        away_teams,
        on="team_a",
        how="left",
        validate="many_to_one",
    )
)

fixtures["kickoff_time"] = pd.to_datetime(
    fixtures["kickoff_time"],
    utc=True,
    errors="coerce",
)

# Convert UTC fixture times to mainland Spain time.
fixtures["kickoff_time_spain"] = (
    fixtures["kickoff_time"]
    .dt.tz_convert("Europe/Madrid")
)

In [10]:
preferred_fixture_columns = [
    "id",
    "event",
    "kickoff_time",
    "kickoff_time_spain",
    "home_team",
    "away_team",
    "team_h_score",
    "team_a_score",
    "team_h_difficulty",
    "team_a_difficulty",
    "started",
    "finished",
]

available_fixture_columns = [
    column
    for column in preferred_fixture_columns
    if column in fixtures.columns
]

fixtures = fixtures[available_fixture_columns].copy()

display(
    fixtures
    .sort_values("kickoff_time_spain")
    .head(20)
)

,id,event,kickoff_time,kickoff_time_spain,home_team,away_team,team_h_score,team_a_score,team_h_difficulty,team_a_difficulty,started,finished
0,1,1,2026-08-21 19:00:00+00:00,2026-08-21 21:00:00+02:00,Arsenal,Coventry City,None,None,2,5,False,False
1,4,1,2026-08-22 11:30:00+00:00,2026-08-22 13:30:00+02:00,Hull City,Man Utd,None,None,4,2,False,False
2,3,1,2026-08-22 14:00:00+00:00,2026-08-22 16:00:00+02:00,Everton,Crystal Palace,None,None,3,3,False,False
3,5,1,2026-08-22 14:00:00+00:00,2026-08-22 16:00:00+02:00,Ipswich Town,Sunderland,None,None,2,2,False,False
4,6,1,2026-08-22 14:00:00+00:00,2026-08-22 16:00:00+02:00,Nott'm Forest,Leeds,None,None,2,3,False,False
5,2,1,2026-08-22 16:30:00+00:00,2026-08-22 18:30:00+02:00,Brentford,Spurs,None,None,3,3,False,False
6,7,1,2026-08-23 13:00:00+00:00,2026-08-23 15:00:00+02:00,Brighton,Aston Villa,None,None,3,3,False,False
7,8,1,2026-08-23 13:00:00+00:00,2026-08-23 15:00:00+02:00,Man City,Bournemouth,None,None,3,5,False,False
8,9,1,2026-08-23 15:30:00+00:00,2026-08-23 17:30:00+02:00,Newcastle,Liverpool,None,None,4,3,False,False
9,10,1,2026-08-24 19:00:00+00:00,2026-08-24 21:00:00+02:00,Fulham,Chelsea,None,None,4,3,False,False


### Gameweeks

In [11]:
gameweeks = pd.DataFrame(bootstrap["events"])

gameweeks["deadline_time"] = pd.to_datetime(
    gameweeks["deadline_time"],
    utc=True,
    errors="coerce",
)

gameweeks["deadline_time_spain"] = (
    gameweeks["deadline_time"]
    .dt.tz_convert("Europe/Madrid")
)

gameweek_columns = [
    "id",
    "name",
    "deadline_time_spain",
    "finished",
    "is_previous",
    "is_current",
    "is_next",
    "average_entry_score",
    "highest_score",
]

gameweek_columns = [
    column
    for column in gameweek_columns
    if column in gameweeks.columns
]

display(gameweeks[gameweek_columns])

,id,name,deadline_time_spain,finished,is_previous,is_current,is_next,average_entry_score,highest_score
0,1,Gameweek 1,2026-08-21 19:30:00+02:00,False,False,False,True,0,None
1,2,Gameweek 2,2026-08-28 19:30:00+02:00,False,False,False,False,0,None
2,3,Gameweek 3,2026-09-04 19:30:00+02:00,False,False,False,False,0,None
3,4,Gameweek 4,2026-09-12 14:30:00+02:00,False,False,False,False,0,None
4,5,Gameweek 5,2026-09-18 19:30:00+02:00,False,False,False,False,0,None
5,6,Gameweek 6,2026-10-10 14:30:00+02:00,False,False,False,False,0,None
6,7,Gameweek 7,2026-10-17 14:30:00+02:00,False,False,False,False,0,None
7,8,Gameweek 8,2026-10-24 14:30:00+02:00,False,False,False,False,0,None
8,9,Gameweek 9,2026-10-31 14:30:00+01:00,False,False,False,False,0,None
9,10,Gameweek 10,2026-11-07 14:30:00+01:00,False,False,False,False,0,None


### Validation

In [12]:
assert not players.empty, "The player table is empty."
assert not fixtures.empty, "The fixture table is empty."
assert players["id"].is_unique, "Player IDs are not unique."
assert players["team_name"].notna().all(), "Some players have no mapped team."
assert players["position"].notna().all(), "Some players have no mapped position."

assert players["price"].between(3.0, 20.0).all(), (
    "At least one player price appears invalid."
)

print(f"Players loaded: {len(players):,}")
print(f"Teams loaded: {len(teams):,}")
print(f"Fixtures loaded: {len(fixtures):,}")
print(f"Gameweeks loaded: {len(gameweeks):,}")

print("\nPlayers by position:")
print(players["position"].value_counts())

print("\nPlayers by team:")
print(
    players["team_short_name"]
    .value_counts()
    .sort_index()
)

Players loaded: 555
Teams loaded: 20
Fixtures loaded: 380
Gameweeks loaded: 38

Players by position:
position
MID    244
DEF    183
FWD     68
GKP     60
Name: count, dtype: int64

Players by team:
team_short_name
ARS    27
AVL    27
BHA    32
BOU    25
BRE    26
CHE    31
COV    28
CRY    29
EVE    23
FUL    21
HUL    27
IPS    27
LEE    24
LIV    34
MCI    30
MUN    33
NEW    24
NFO    26
SUN    25
TOT    36
Name: count, dtype: int64
